# Project Sentinel

## Notebook 1 : UPI Transaction Data Generator

#### Objective

###### This notebook generates a realistic UPI transaction dataset in CSV format. The generated dataset  simulates real-world payment transactions by including valid records along with intentionally corrupted data such as null values, duplicates, negative amounts, and inconsistent formatting.

######The generated CSV file will be stored in the Landing layer and will serve as the source dataset for the Bronze layer in the Medallion Architecture.

## STEP 1: Import Libraries

In [0]:
import random
import uuid
import pandas as pd
import copy
from datetime import datetime, timedelta

## STEP 2: Configuration

In [0]:
#Databricks Volume Path
landing_path = "/Volumes/dbacademy/default/myvolume/Project Sentinel"

# Output File Name
output_file = "upi_transactions.csv"

# Number of Transactions
num_transactions = 20000

##STEP 3: Verify Configuration

In [0]:
print("Landing Path :", landing_path)
print("Number of Transactions :", num_transactions)

Landing Path : /Volumes/dbacademy/default/myvolume/Project Sentinel
Number of Transactions : 20000


## STEP 4: Reference Data

#### 4.1 - Banks and UPI Handles

In [0]:
# Bank Names
banks = {
    "SBI": "oksbi",
    "HDFC": "okhdfc",
    "ICICI": "icici",
    "Axis": "axis"
}

#### 4.2 - Cities & States

In [0]:
# Cities and States
locations = {
    "Jaipur": "Rajasthan",
    "Delhi": "Delhi",
    "Mumbai": "Maharashtra",
    "Pune": "Maharashtra",
    "Ahmedabad": "Gujarat",
    "Indore": "Madhya Pradesh",
    "Bangalore": "Karnataka",
    "Hyderabad": "Telangana"
}

#### 4.3 - User Names

In [0]:
# User Names
user_names = [
    "rahul","priya","amit","neha","rohit","pooja","arjun","kapil","vikas","sneha","prachi","riya","manish",
    "nisha","ankit","kartik","simran","karan","divya","harsh"
]

#### 4.4 - Transaction Types

In [0]:
# Transaction Types
transaction_types = [
    "P2P",
    "Merchant",
    "Recharge",
    "Bill Payment"
]

#### 4.5 - Transaction Status

In [0]:
# Transaction Status
status_list = [
    "SUCCESS",
    "FAILED"
]

#### 4.6 - Failure Reasons

In [0]:
# Failure Reasons
failure_reasons = [
    None,
    "Timeout",
    "Bank Server Down",
    "Invalid UPI",
    "Daily Limit Exceeded"
]

#### 4.7 Device ID's

In [0]:
#Device IDs
device_ids = [
    f"DEV{100000 + i}"
    for i in range(200)
]

#### 4.8 - Verify Reference Data

In [0]:
print("Banks :",banks)
print("\nLocations :",locations)
print("\nTotal User Names :",len(user_names))
print("\nTransaction Types :",transaction_types)
print("\nStatus :",status_list)
print("\nFailure Reasons :",failure_reasons)

Banks : {'SBI': 'oksbi', 'HDFC': 'okhdfc', 'ICICI': 'icici', 'Axis': 'axis'}

Locations : {'Jaipur': 'Rajasthan', 'Delhi': 'Delhi', 'Mumbai': 'Maharashtra', 'Pune': 'Maharashtra', 'Ahmedabad': 'Gujarat', 'Indore': 'Madhya Pradesh', 'Bangalore': 'Karnataka', 'Hyderabad': 'Telangana'}

Total User Names : 20

Transaction Types : ['P2P', 'Merchant', 'Recharge', 'Bill Payment']

Status : ['SUCCESS', 'FAILED']

Failure Reasons : [None, 'Timeout', 'Bank Server Down', 'Invalid UPI', 'Daily Limit Exceeded']


## Step 5 – Generate Transaction Function

In [0]:
def generate_transaction():
    transaction = {}
    #Transaction ID
    transaction["transaction_id"] = str(uuid.uuid4())

    #Transaction Timestamp
    random_days = random.randint(0, 29)
    random_seconds = random.randint(0, 86400)
    transaction_time = datetime.now() - timedelta(days=random_days,seconds=random_seconds)

    transaction["transaction_timestamp"] = transaction_time.strftime("%Y-%m-%d %H:%M:%S")

    #Sender Bank
    sender_bank = random.choice(list(banks.keys()))
    transaction["sender_bank"] = sender_bank

    #Receiver Bank
    receiver_bank = random.choice([x for x in banks if x != sender_bank])
    transaction["receiver_bank"] = receiver_bank

    #Sender UPI
    sender_name = random.choice(user_names)
    transaction["sender_upi"] = (f"{sender_name}{random.randint(100,999)}@{banks[sender_bank]}")

    #Receiver UPI
    receiver_name = random.choice(user_names)
    transaction["receiver_upi"] = (f"{receiver_name}{random.randint(100,999)}@{banks[receiver_bank]}")

    # Transaction Amount
    transaction["amount"] = random.randint(1, 50000)

    # Transaction Type
    transaction["transaction_type"] = random.choices(transaction_types,weights=[50,20,15,15],k=1)[0]

    # Transaction Status
    transaction["transaction_status"] = random.choices(status_list,weights=[97, 3],k=1)[0]

    #City & State
    city = random.choice(list(locations.keys()))
    transaction["city"] = city
    transaction["state"] = locations[city]

    #Device ID
    transaction["device_id"] = random.choice(device_ids)

    # Response Time
    transaction["response_time_ms"] = random.randint(100,3000)
        
    # Failure Reason
    if transaction["transaction_status"] == "FAILED":
        transaction["failure_reason"] = random.choice(failure_reasons)
            
    else:
        transaction["failure_reason"] = None

    return transaction

### 5.1 Test the Function

In [0]:
sample_transaction = generate_transaction()

sample_transaction

{'transaction_id': '5a199524-8676-41ba-acb0-ffef65f04f50',
 'transaction_timestamp': '2026-07-03 03:14:01',
 'sender_bank': 'HDFC',
 'receiver_bank': 'ICICI',
 'sender_upi': 'arjun574@okhdfc',
 'receiver_upi': 'ankit793@icici',
 'amount': 28038,
 'transaction_type': 'P2P',
 'transaction_status': 'SUCCESS',
 'city': 'Indore',
 'state': 'Madhya Pradesh',
 'device_id': 'DEV100158',
 'response_time_ms': 2109,
 'failure_reason': None}

## Step 6 – Inject Bad Data

In [0]:
def inject_bad_data(transaction):

    # Inject Null Amount (3%)
    if random.random() < 0.03:
        transaction["amount"] = None

    # Inject Negative Amount (2%)
    if random.random() < 0.02:
        if transaction["amount"] is not None:
            transaction["amount"] = -transaction["amount"]

    # Inject Extra Spaces (2%)
    if random.random() < 0.02:
        transaction["sender_bank"] = " " + transaction["sender_bank"] + " "

     # Inject Wrong Data Type (1%)
    if random.random() < 0.01:
        if transaction["amount"] is not None:
            transaction["amount"] = str(transaction["amount"])

    return transaction

### 6.1 Test the Function

In [0]:
transaction = generate_transaction()

print("Before Injection")
print(transaction)

transaction = inject_bad_data(transaction)

print("\nAfter Injection")
print(transaction)

Before Injection
{'transaction_id': 'c3a80ae1-d09f-4f91-8886-20d3db28b8bf', 'transaction_timestamp': '2026-07-07 21:05:07', 'sender_bank': 'ICICI', 'receiver_bank': 'Axis', 'sender_upi': 'divya566@icici', 'receiver_upi': 'prachi589@axis', 'amount': 45219, 'transaction_type': 'Merchant', 'transaction_status': 'SUCCESS', 'city': 'Ahmedabad', 'state': 'Gujarat', 'device_id': 'DEV100163', 'response_time_ms': 931, 'failure_reason': None}

After Injection
{'transaction_id': 'c3a80ae1-d09f-4f91-8886-20d3db28b8bf', 'transaction_timestamp': '2026-07-07 21:05:07', 'sender_bank': 'ICICI', 'receiver_bank': 'Axis', 'sender_upi': 'divya566@icici', 'receiver_upi': 'prachi589@axis', 'amount': -45219, 'transaction_type': 'Merchant', 'transaction_status': 'SUCCESS', 'city': 'Ahmedabad', 'state': 'Gujarat', 'device_id': 'DEV100163', 'response_time_ms': 931, 'failure_reason': None}


## Step 7 Generate 20,000 Transactions

In [0]:
transactions = []

for i in range(num_transactions):
    transaction = generate_transaction()
    transaction = inject_bad_data(transaction)
    transactions.append(transaction)

In [0]:
#Check Dataset Size
print(f"Total Transactions Generated : {len(transactions)}")

Total Transactions Generated : 20000


## Step 8 – Inject Duplicate Records

In [0]:
# Number of Duplicate Records
duplicate_count = int(num_transactions * 0.02)

print(f"Duplicate Records to Add : {duplicate_count}")

Duplicate Records to Add : 400


In [0]:
# Inject Duplicate Records

for i in range(duplicate_count):
    duplicate_record = copy.deepcopy(random.choice(transactions))
    transactions.append(duplicate_record)

In [0]:
#Verify Dataset Size
print(f"Final Dataset Size : {len(transactions)}")

Final Dataset Size : 20400


## Step 9 – Create DataFrame

In [0]:
df = pd.DataFrame(transactions)
df.head()

,transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason
0,2bbc7344-a5dc-4ffb-bff0-171c1c7620c0,2026-07-08 14:04:52,HDFC,SBI,ankit557@okhdfc,rohit684@oksbi,7090,P2P,SUCCESS,Indore,Madhya Pradesh,DEV100111,2366,None
1,61f432e0-bd55-4cfe-b14d-2beec8fd638f,2026-06-15 06:45:32,ICICI,HDFC,priya403@icici,manish738@okhdfc,15597,P2P,SUCCESS,Mumbai,Maharashtra,DEV100174,1016,None
2,bb370089-0c45-4264-bc56-15f741cbef38,2026-06-25 21:59:21,ICICI,SBI,manish200@icici,ankit279@oksbi,40824,Merchant,SUCCESS,Jaipur,Rajasthan,DEV100122,217,None
3,62fb68f6-15e6-4ee9-9478-f772f83d9e33,2026-06-18 15:32:26,ICICI,HDFC,neha849@icici,arjun756@okhdfc,18156,Recharge,SUCCESS,Hyderabad,Telangana,DEV100003,473,None
4,79b0fb85-a310-48a2-957b-92e38f6e789f,2026-07-06 08:11:01,HDFC,Axis,rohit687@okhdfc,arjun923@axis,24257,P2P,SUCCESS,Pune,Maharashtra,DEV100140,2425,None


In [0]:
print("Shape of Dataset :", df.shape)

Shape of Dataset : (20400, 14)


## Step 10 – Save CSV

In [0]:
output_path = f"{landing_path}/{output_file}"

df.to_csv(output_path,index=False)
print("Dataset Successfully Saved")

print(output_path)

Dataset Successfully Saved
/Volumes/dbacademy/default/myvolume/Project Sentinel/upi_transactions.csv


## Step 11 – Final Verification

In [0]:
df.sample(10)

,transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason
7299,45c20fa0-595e-4ca8-9622-2693e39e398d,2026-06-28 16:28:46,ICICI,HDFC,karan222@icici,ankit436@okhdfc,21471,P2P,SUCCESS,Jaipur,Rajasthan,DEV100109,730,None
7897,91cd330e-9fe5-4073-9eb4-edd490e1c166,2026-06-19 08:28:17,HDFC,SBI,priya467@okhdfc,priya331@oksbi,38479,P2P,SUCCESS,Indore,Madhya Pradesh,DEV100115,1225,None
3130,40d3816a-4b48-4e52-a8e6-88c1fd781ef8,2026-06-16 05:04:42,ICICI,HDFC,sneha140@icici,nisha580@okhdfc,18843,Merchant,SUCCESS,Pune,Maharashtra,DEV100035,2533,None
13280,46c65dfa-90b9-4515-8cba-88efc46a3557,2026-07-01 12:23:33,HDFC,Axis,harsh244@okhdfc,nisha863@axis,28865,P2P,SUCCESS,Jaipur,Rajasthan,DEV100026,301,None
8568,6156d83e-c686-4ba2-9c38-cbc78086eb94,2026-06-20 08:42:47,HDFC,ICICI,nisha756@okhdfc,riya254@icici,49879,Merchant,SUCCESS,Pune,Maharashtra,DEV100187,2069,None
7519,5ca65ad5-9dd3-473a-be71-2ecb21cea598,2026-06-19 16:37:37,ICICI,SBI,sneha502@icici,kartik910@oksbi,30225,P2P,SUCCESS,Hyderabad,Telangana,DEV100033,1068,None
416,2a867006-3079-4733-9c54-dc9d332a02f7,2026-07-07 22:35:52,ICICI,Axis,vikas431@icici,simran866@axis,8112,Bill Payment,SUCCESS,Mumbai,Maharashtra,DEV100028,2709,None
5184,57c41709-d681-4c4c-8cc7-e26a9d3080e3,2026-07-12 10:24:01,Axis,SBI,nisha523@axis,nisha790@oksbi,38800,P2P,SUCCESS,Hyderabad,Telangana,DEV100021,2318,None
147,985a613b-cfda-4f74-995d-1a8c9174b6fb,2026-06-12 21:59:45,SBI,HDFC,priya798@oksbi,rohit540@okhdfc,46569,P2P,SUCCESS,Ahmedabad,Gujarat,DEV100155,2321,None
15091,cd9ff8d6-d010-4e55-92cf-b6304f3b4219,2026-06-30 00:35:42,HDFC,SBI,vikas838@okhdfc,harsh812@oksbi,23373,P2P,SUCCESS,Hyderabad,Telangana,DEV100125,1913,None
